# 한국어 요약 서비스 — 실행 / 테스트 노트북

이 노트북은 프로젝트를 **처음부터 끝까지 실행하고 테스트**합니다. 위에서부터 셀을 순서대로 실행하세요.

**요약 방식 3가지**
| 모드 | 모델 | 특징 |
|---|---|---|
| ⚡ 빠름 (fast) | t5-base | 1~3초, 로컬, 무료 |
| 🎯 정확 (accurate) | Qwen2.5-3B | CPU에서 1~3분, 로컬, 무료 |
| 🏆 최고 (cloud) | Gemini 2.5 Flash | 빠르고 품질 좋음, **무료 등급 API 키 필요** |

마지막엔 **Gemini가 세 요약을 채점·비교**합니다.

> ⚠️ 코드(app/*.py)를 고쳤다면 **커널 재시작(Kernel → Restart) 후 맨 위부터** 다시 실행하세요. 메모리에 남은 옛 코드가 반영을 막을 수 있습니다.

## 1. 환경 확인

**개념** — 이 노트북이 올바른 파이썬(가상환경 `.venv`)과 폴더에서 도는지 확인합니다.
`app/` 폴더가 보여야 서버 코드를 불러올 수 있습니다.

In [ ]:
import sys, os                       # sys: 파이썬 정보, os: 폴더/환경변수 다루기

print("python:", sys.executable)       # .venv 안의 python.exe 경로여야 정상
print("작업 폴더:", os.getcwd())        # project_01 폴더여야 함
print("app 폴더 있음:", os.path.isdir("app"))  # True 여야 함 (서버 코드 위치)

## 2. 서버 실행 도우미 정의

**개념** — 보통 서버는 터미널에서 `uvicorn`으로 켭니다. 여기서는 노트북 안에서 켜고 끄려고,
서버를 **백그라운드 스레드**(다른 작업과 동시에 도는 실행 줄기)에서 띄우는 도우미를 만듭니다.
- `serve_in_thread("app.main:app")` : 서버 켜기
- `stop_server(8000)` : 서버 끄기

이 셀은 **한 번만** 실행하면 됩니다.

In [ ]:
import asyncio, threading, time, socket, contextlib   # 비동기/스레드/대기/포트확인 도구
import uvicorn                                            # FastAPI 앱을 실행하는 웹서버

for _d in ("app", "models", "data", "frontend"):          # 필요한 폴더가 없으면 만들어 둔다
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}                                             # 포트 -> (서버, 스레드) 보관소

def _port_open(host, port):                               # 해당 포트에 서버가 떠 있는지 검사
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0            # 0이면 연결됨(서버 있음)

def stop_server(port=8000):                               # 실행 중인 서버를 멈춘다
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True                             # uvicorn에 "종료해" 신호
    for _ in range(50):                                   # 스레드가 끝날 때까지 잠깐 대기
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host="127.0.0.1", port=8000, log_level="warning"):
    """백그라운드 스레드에서 uvicorn 서버를 띄운다. app 예: 'app.main:app'."""
    stop_server(port)                                     # 같은 포트에 떠 있던 서버는 먼저 종료
    if _port_open(host, port):
        print(f"⚠️ 포트 {port}를 다른 프로세스가 쓰는 중입니다. 종료 후 다시 실행하세요.")
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(":")[0], None)          # 파일을 다시 저장했으면 최신 코드 반영
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop="asyncio")
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None          # 스레드에서는 신호 처리 끔
    def _run():                                            # 스레드가 실제로 돌릴 함수
        loop = asyncio.SelectorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())            # 서버 시작(모델 로드 포함)
    thread = threading.Thread(target=_run, daemon=True)    # 데몬 스레드(노트북 종료 시 같이 종료)
    thread.start()
    _SERVERS[port] = (server, thread)
    for i in range(600):                                   # 최대 5분간 서버가 뜰 때까지 대기
        if _port_open(host, port):
            print(f"✅ 서버 실행됨: http://{host}:{port}")
            return server
        if not thread.is_alive():
            print("❌ 서버 스레드가 종료됐습니다. 로그를 확인하세요.")
            return server
        if i > 0 and i % 20 == 0:
            print(f"  ... 모델 로드 중 ({i//2}초 경과)")
        time.sleep(0.5)
    print("⏱️ 5분 내 시작 실패. 로그를 확인하세요.")
    return server

print("도우미 준비 완료 (serve_in_thread, stop_server)")

## 3. (선택) 빠름 모델 단독 테스트

**개념** — 서버 없이 t5 요약 모델만 불러와 동작을 확인합니다.
`pipeline("summarization", ...)` 은 Hugging Face가 모델을 손쉽게 쓰게 해주는 도구입니다.
이 t5 모델은 입력 앞에 **`summarize: `** 를 붙여야 품질이 좋습니다(공식 사용법).
처음 실행 시 모델을 내려받아 시간이 걸릴 수 있습니다.

In [ ]:
from transformers import pipeline                       # 모델을 간단히 불러오는 도구

summarizer = pipeline("summarization", model="eenzeenee/t5-base-korean-summarization")

sample = (
    "오늘 회의에서는 프로젝트 일정 지연 원인과 대응 방안에 대해 논의하였다. "
    "주요 지연 원인은 부품 입고 일정 변경, 설계 검토 지연, 일부 제작 공정의 병목으로 확인되었다. "
    "이에 따라 관련 부서에서는 납기 영향도를 재검토하고, 고객사에는 변경 가능성이 있는 일정에 대해 사전 안내하기로 하였다."
)
out = summarizer("summarize: " + sample,                  # 접두어 + 원문
                 max_length=120, min_length=30,           # 요약 길이 범위(토큰)
                 do_sample=False,                         # 매번 같은 결과
                 truncation=True,                         # 너무 길면 자동으로 잘라 길이 경고 방지
                 num_beams=6, no_repeat_ngram_size=3, length_penalty=2.0)  # 품질 옵션
print(out[0]["summary_text"])

## 4. FastAPI 서버 실행

**개념** — `app/main.py` 의 서버를 띄웁니다. 서버가 켜질 때 **빠름(t5) 모델을 1번 로드**합니다.
정확(Qwen)·최고(Gemini)는 무거우므로 **첫 요청이 올 때** 준비됩니다(지연 로드).

In [ ]:
serve_in_thread("app.main:app", port=8000)

## 5. 테스트 준비 (주소 · 인증 · Gemini 키)

**개념** — 모든 요청에는 서비스 인증용 헤더 `X-API-Key: my-secret-key` 가 필요합니다.
🏆 최고 모드와 평가는 **Gemini API 키**가 있어야 동작합니다(무료 등급).
키는 https://aistudio.google.com 에서 무료로 발급받아 아래 `GEMINI_KEY` 에 붙여넣으세요.
(비워두면 빠름·정확 모드만 테스트됩니다.)

In [ ]:
import requests, time, json                             # HTTP 요청 / 시간측정 / JSON

API = "http://127.0.0.1:8000"                             # 서버 주소
KEY = {"X-API-Key": "my-secret-key"}                      # 서비스 인증 헤더

GEMINI_KEY = ""   # ← 여기에 무료 Gemini 키를 붙여넣으세요 (없으면 빈 문자열로 두기)

sample = (
    "오늘 주간 회의에서는 신제품 출시 일정과 마케팅 예산 배분에 대해 논의하였다. "
    "개발팀은 핵심 기능 구현이 예정보다 일주일 지연되고 있다고 보고하였고, "
    "디자인팀은 최종 시안이 다음 주 화요일에 완료된다고 공유하였다. "
    "마케팅팀은 온라인 광고 비중을 늘리는 대신 오프라인 행사 예산을 축소하는 방안을 제안하였다. "
    "최종적으로 출시일은 2주 늦추되, 사전 예약 이벤트를 먼저 진행하기로 결정하였다."
)
print("준비 완료. GEMINI_KEY 설정 여부:", bool(GEMINI_KEY))

### 5-1. /health — 서버 상태 확인

**개념** — 서버와 각 모델 이름이 잘 잡혔는지 확인하는 점검용 엔드포인트(인증 불필요).

In [ ]:
r = requests.get(f"{API}/health")                       # GET 요청(상태 확인)
print(r.status_code)                                     # 200이면 정상
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

### 5-2. API Key 없이 호출 → 401

**개념** — 인증 키 없이 `/predict` 를 부르면 거부됩니다(401 Unauthorized).
`app/auth.py` 의 `verify_api_key` 가 막아줍니다.

In [ ]:
r = requests.post(f"{API}/predict", json={"text": sample})   # 헤더(KEY) 없이 호출
print("상태:", r.status_code)                                 # 401 기대
print(r.json())

### 5-3. 너무 짧은 입력 → 422

**개념** — 입력이 30자 미만이면 Pydantic 검증(`app/schemas.py`)이 막아 422를 돌려줍니다.
잘못된 입력이 모델에 닿기 전에 차단됩니다.

In [ ]:
r = requests.post(f"{API}/predict", json={"text": "너무 짧은 글"}, headers=KEY)
print("상태:", r.status_code)                                 # 422 기대
print(json.dumps(r.json(), ensure_ascii=False, indent=2)[:300])

### 5-4. ⚡ 빠름(fast) 모드 → 200

**개념** — t5 모델로 1~3초 만에 요약. `max_length`/`min_length` 를 안 보내면
입력 길이에 맞춰 **자동**으로 길이를 정합니다(짧은 글은 짧게, 긴 글은 길게).

In [ ]:
t0 = time.time()
r = requests.post(f"{API}/predict", json={"text": sample, "mode": "fast"}, headers=KEY)
print("상태:", r.status_code, f"/ {time.time()-t0:.1f}초")
d = r.json()
print("모델:", d["model_name"])
print("요약:", d["summary"])

### 5-5. 🎯 정확(accurate) 모드 → 200

**개념** — 로컬 LLM(Qwen2.5-3B). 품질은 좋지만 **CPU라 1~3분** 걸립니다.
서버는 `run_in_executor` 로 추론을 별도 스레드에서 돌려, 그동안에도 멈추지 않습니다.

In [ ]:
t0 = time.time()
r = requests.post(f"{API}/predict", json={"text": sample, "mode": "accurate"}, headers=KEY, timeout=600)
print("상태:", r.status_code, f"/ {time.time()-t0:.0f}초")
print("요약:", r.json()["summary"])

### 5-6. 🏆 최고(cloud) 모드 → 200  *(Gemini 키 필요)*

**개념** — 클라우드 LLM(Gemini 2.5 Flash). 빠르고 품질 좋음.
요청 본문에 `gemini_api_key` 를 함께 보내면 그 키로 호출합니다(없으면 서버 환경변수 사용).

In [ ]:
payload = {"text": sample, "mode": "cloud"}
if GEMINI_KEY:
    payload["gemini_api_key"] = GEMINI_KEY                # 위에서 설정한 키 사용
t0 = time.time()
r = requests.post(f"{API}/predict", json=payload, headers=KEY, timeout=120)
print("상태:", r.status_code, f"/ {time.time()-t0:.1f}초")
print(r.json().get("summary") or r.json())               # 키 없으면 안내 메시지가 보임

### 5-7. 🔑 Gemini 키 확인

**개념** — 키가 유효한지 가볍게 확인합니다(모델 목록 조회 — 생성 토큰을 쓰지 않음).

In [ ]:
r = requests.post(f"{API}/verify_key", json={"gemini_api_key": GEMINI_KEY}, headers=KEY)
print(r.json())                                          # {'valid': True/False, 'detail': ...}

## 6. 3가지 방식 비교 + Gemini 평가

**개념** — 같은 입력을 세 방식으로 모두 요약한 뒤, 그 요약들을 `/evaluate` 로 보내면
**Gemini가 4개 항목(정확성·핵심포착·간결성·자연스러움)을 5점 만점으로 채점**하고
가장 좋은 요약을 골라줍니다. (최고 모드·평가는 Gemini 키가 있어야 동작)

In [ ]:
# 1) 세 방식으로 각각 요약 (성공한 것만 모음)
summaries = {}
for label, mode in [("빠름", "fast"), ("정확", "accurate"), ("최고", "cloud")]:
    p = {"text": sample, "mode": mode}
    if mode == "cloud" and GEMINI_KEY:
        p["gemini_api_key"] = GEMINI_KEY
    t0 = time.time()
    r = requests.post(f"{API}/predict", json=p, headers=KEY, timeout=600)
    if r.status_code == 200:
        s = r.json()["summary"]
        summaries[label] = s
        print(f"[{label}] {time.time()-t0:.0f}초 / {len(s)}자: {s[:60]}...")
    else:
        print(f"[{label}] 실패({r.status_code}): {str(r.json())[:80]}")

# 2) Gemini 평가 (성공한 요약이 2개 이상일 때)
if len(summaries) >= 2:
    ep = {"text": sample, "summaries": summaries}
    if GEMINI_KEY:
        ep["gemini_api_key"] = GEMINI_KEY
    r = requests.post(f"{API}/evaluate", json=ep, headers=KEY, timeout=180)
    data = r.json()
    print("\n=== Gemini 평가 ===")
    for e in data.get("evaluations", []):                # 방식별 항목 점수
        print(f"[{e.get('name')}] 정확성 {e.get('정확성')}/핵심포착 {e.get('핵심포착')}/"
              f"간결성 {e.get('간결성')}/자연스러움 {e.get('자연스러움')} — {e.get('총평','')}")
    print("\n🏅 가장 좋은 요약:", data.get("best"))
    print("이유:", data.get("reason"))
else:
    print("\n평가하려면 요약이 2개 이상 성공해야 합니다 (Gemini 키 확인).")

## 7. 서버 종료

**개념** — 테스트가 끝나면 서버를 멈춥니다(메모리 해제).

In [ ]:
stop_server(8000)
print("서버 종료됨")